# 饲喂料量vs换料vs标准

## README

## 4栋2单元数据

In [ ]:
# ------------- 配置数据 ------------------
WEANING_WEIGHT = 6.5  # 断奶仔猪重

In [ ]:
# 读取数据
import pandas as pd
from feed_analysis.config.path import PATH_DATA, PATH_FEED_PROCESSED, PATH_FIGURE_HTML
from feed_analysis.config.coding_schema import STD_HEADER_NAME
from feed_analysis.feed_pipeline.utils.age import recalibrate_age

# # 更新日龄
# recalibrate_age(PATH_FEED_PROCESSED/'育肥4-2_column.parquet', reference_date='2025-8-25', reference_age=23)         # 根据杨乐乐主管1.4提供信息
# recalibrate_age(PATH_FEED_PROCESSED/'育肥4-2_build.parquet', reference_date='2025-8-25', reference_age=23)         # 根据杨乐乐主管1.4提供信息

# 读取喂食量数据 
df_42_col = pd.read_parquet(PATH_FEED_PROCESSED/'育肥4-2_column.parquet', engine='pyarrow')
df_42_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥4-2_build.parquet', engine='pyarrow')

# 读取存栏量数据
df_42_num = pd.read_excel(PATH_DATA / 'ori' / '育肥4-2单元存栏数.xlsx', header=3).rename(columns=STD_HEADER_NAME).sort_values(by=['Date'])
df_42_num['Date'] = pd.to_datetime(df_42_num['Date']).dt.date

# 计算头均值
df_42_build['stock_num'] = df_42_build['Date'].map(df_42_num.set_index('Date')['stock_num'])
df_42_build.dropna(subset=['stock_num'], inplace=True)
df_42_build['avg_food_kg'] = df_42_build['food_total_kg'] / df_42_build['stock_num']

# 插补缺失值
from feed_analysis.feed_pipeline.utils.interpolate import interpolation
df_42_build = interpolation(df_42_build, index_col='age', )

In [ ]:
from itables import show
show(df_42_build)

Loading ITables v2.6.2 from the internet... (need help?)


In [ ]:
# -------------- 读取标准日龄FCR数据 ------------------
df_hx_std = pd.read_excel(PATH_DATA / 'ori' / '汇兴牧业-标准日龄饲料.xlsx', header=1)   # 汇兴2026饲喂标注

# ------------ 添加日龄和料肉比---------------
df_42_build['day_fcr'] = df_42_build['age'].map(df_hx_std.set_index('日龄')['日料肉比'])
# 填补缺失值
df_42_build.loc[df_42_build['day_fcr'].isna(), 'day_fcr'] = df_42_build['day_fcr'].min()

# ----------- 添加日增重和估计体重等---------------
df_42_build['day_weight'] = df_42_build['avg_food_kg'] / df_42_build['day_fcr']     # 日增重（kg）
df_42_build['weight_cumsum'] = df_42_build['day_weight'].cumsum()                   # 累计日增重（kg）
df_42_build['weight'] = df_42_build['weight_cumsum'] + WEANING_WEIGHT               # 总计重量
df_42_build['day_pct_weight'] = df_42_build['day_weight'] / df_42_build['weight']     # 日增重百分比


总趋势变化

In [ ]:
from feed_analysis.visualization.line_plot import line_dynamic_single
line_dynamic_single(df_42_build, 'avg_food_kg', '育肥4-2单元料头均值', color='red')

In [ ]:
from feed_analysis.visualization.line_plot import line_dynamic_single
line_dynamic_single(df_42_build, 'day_weight', '育肥4-2单元日增重', color='red')

In [ ]:
from feed_analysis.visualization.line_plot import line_dynamic_single
line_dynamic_single(df_42_build, 'day_pct_weight', '育肥4-2单元日增重百分比', color='red')

In [ ]:
from feed_analysis.visualization.line_plot import line_dynamic_single
line_dynamic_single(df_42_build, 'day_weight', '育肥4-2单元日增重', color='red')

### 日龄-总重拟合 - Gompertz模型

In [ ]:
from feed_analysis.growing_fit.gompertz import fit_gompertz, gompertz_pred, fit_gompertz_fixed_L, fit_gompertz_fixed_x0

x = df_42_build['age'].values
y = df_42_build['weight'].values
# gompertz_params_42 = fit_gompertz(x, y)     # Gompertz模型拟合(直接最小二乘)
gompertz_params_42 = fit_gompertz_fixed_L(x, y, 180)     # Gompertz模型拟合，固定L，R2最大可达0.7
# gompertz_params_42 = fit_gompertz_fixed_x0(x, y, x0=150)     # Gompertz模型拟合(直接最小二乘)
print(gompertz_params_42)

# 添加拟合结果
df_42_build['weight_fit'] = gompertz_pred(df_42_build['age'].values, gompertz_params_42)  # 拟合结果
df_42_build['day_weight_fit'] = df_42_build['weight_fit'].diff()  # 日增重拟合结果
df_42_build.dropna(inplace=True)  # 日增重拟合结果

# df_42_build.to_excel(PATH_DATA / 'growth' / '汇兴4栋生长数据-gompertz拟合.xlsx', index=False)

In [ ]:
df_42_build.head()

In [ ]:
# ------------- 模型拟合评估 -------------
# 体重
from feed_analysis.growing_fit.metrics import get_r2, get_rmse, get_mape
r2_weight = get_r2(df_42_build['weight'], df_42_build['weight_fit'])
rmse_weight = get_rmse(df_42_build['weight'], df_42_build['weight_fit'])
mape_weight = get_mape(df_42_build['weight'], df_42_build['weight_fit'])
# 日增重
r2_day_weight = get_r2(df_42_build['day_weight'], df_42_build['day_weight_fit'])
rmse_day_weight = get_rmse(df_42_build['day_weight'], df_42_build['day_weight_fit'])
mape_day_weight = get_mape(df_42_build['day_weight'], df_42_build['day_weight_fit'])

print(f"体重拟合评估：R2={r2_weight:.4f}, RMSE={rmse_weight:.4f}, MAPE={mape_weight:.4f}")
print(f"日增重拟合评估：R2={r2_day_weight:.4f}, RMSE={rmse_day_weight:.4f}, MAPE={mape_day_weight:.4f}")

In [ ]:
# df_42_build.to_excel(PATH_DATA / 'growth' / '汇兴4栋生长数据-log-log拟合.xlsx', index=False)
from feed_analysis.growing_fit.log_log_fit import plot_true_pred
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['weight'], mode='markers', name='实际重量', opacity=0.5))
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['weight_fit'], mode='lines', name='拟合重量', opacity=1))

change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green", annotation=dict(text='换料'))   

fig.update_layout(title='育肥4栋2单元日增重拟合') 
# fig.write_html(PATH_FIGURE_HTML/'育肥4栋2单元总重拟合.html')
fig.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['day_weight'], mode='markers', name='实际日增重'))
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['day_weight_fit'], mode='lines', name='拟合日增重'))

change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green", annotation=dict(text='换料'))   

fig.update_layout(title='育肥4栋2单元日增重拟合')

### 日龄-总重拟合 - Logistic模型

In [ ]:
from feed_analysis.growing_fit.logistic import fit_logistic, fit_logistic_fixed_L, logistic_pred

x = df_42_build['age'].values
y = df_42_build['weight'].values
logistic_params_42 = fit_logistic(x, y)     # Logistic模型拟合(直接最小二乘)
# logistic_params_42 = fit_logistic_fixed_L(x, y, 180)     # Logistic模型拟合，固定L，R2最大可达
print(logistic_params_42)

# 添加拟合结果
df_42_build['weight_fit'] = logistic_pred(df_42_build['age'].values, logistic_params_42)  # 拟合结果
df_42_build['day_weight_fit'] = df_42_build['weight_fit'].diff()  # 日增重拟合结果
df_42_build.dropna(inplace=True)  # 日增重拟合结果

# df_42_build.to_excel(PATH_DATA / 'growth' / '汇兴4栋生长数据-gompertz拟合.xlsx', index=False)

In [ ]:
# ------------- 模型拟合评估 -------------
# 体重
from feed_analysis.growing_fit.metrics import get_r2, get_rmse, get_mape
r2_weight = get_r2(df_42_build['weight'], df_42_build['weight_fit'])
rmse_weight = get_rmse(df_42_build['weight'], df_42_build['weight_fit'])
mape_weight = get_mape(df_42_build['weight'], df_42_build['weight_fit'])
# 日增重
r2_day_weight = get_r2(df_42_build['day_weight'], df_42_build['day_weight_fit'])
rmse_day_weight = get_rmse(df_42_build['day_weight'], df_42_build['day_weight_fit'])
mape_day_weight = get_mape(df_42_build['day_weight'], df_42_build['day_weight_fit'])

print(f"体重拟合评估：R2={r2_weight:.4f}, RMSE={rmse_weight:.4f}, MAPE={mape_weight:.4f}")
print(f"日增重拟合评估：R2={r2_day_weight:.4f}, RMSE={rmse_day_weight:.4f}, MAPE={mape_day_weight:.4f}")

In [ ]:
# df_42_build.to_excel(PATH_DATA / 'growth' / '汇兴4栋生长数据-log-log拟合.xlsx', index=False)
from feed_analysis.growing_fit.log_log_fit import plot_true_pred
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['weight'], mode='markers', name='实际重量', opacity=0.5))
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['weight_fit'], mode='lines', name='拟合重量', opacity=1))

change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green", annotation=dict(text='换料'))   

fig.update_layout(title='育肥4栋2单元日增重拟合') 
# fig.write_html(PATH_FIGURE_HTML/'育肥4栋2单元总重拟合.html')
fig.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['day_weight'], mode='markers', name='实际日增重'))
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['day_weight_fit'], mode='lines', name='拟合日增重'))

change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green", annotation=dict(text='换料'))   

fig.update_layout(title='育肥4栋2单元日增重拟合')

### 日龄-总重拟合 - 多项式模型

In [ ]:
from feed_analysis.growing_fit.log_log_fit import fit_poly_log_log, poly_log_pred

x = df_42_build['age'].values
y = df_42_build['weight'].values
df_42_build, poly_log_model, poly, scaler = fit_poly_log_log(df=df_42_build, x_col="age", y_col="weight", degree=4)     # Log模型拟合(直接最小二乘)
# print(poly_log_params_42)

# 添加拟合结果
df_pred = poly_log_pred(df_42_build['age'].values, poly_log_model, poly, scaler)  # 拟合结果
df_42_build['weight_fit'] = df_42_build['age'].map(df_pred.set_index('x')['y_pred'])  # 拟合结果
df_42_build['day_weight_fit'] = df_42_build['weight_fit'].diff()  # 日增重拟合结果
df_42_build.dropna(inplace=True)  # 日增重拟合结果

# df_42_build.to_excel(PATH_DATA / 'growth' / '汇兴4栋生长数据-gompertz拟合.xlsx', index=False)

                            OLS Regression Results                            
Dep. Variable:                 weight   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 2.218e+05
Date:                Tue, 20 Jan 2026   Prob (F-statistic):          6.67e-207
Time:                        16:55:10   Log-Likelihood:                 374.24
No. Observations:                 111   AIC:                            -738.5
Df Residuals:                     106   BIC:                            -724.9
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.6703      0.001   4548.171      0.0

In [ ]:
# ------------- 模型拟合评估 -------------
# 体重
from feed_analysis.growing_fit.metrics import get_r2, get_rmse, get_mape
r2_weight = get_r2(df_42_build['weight'], df_42_build['weight_fit'])
rmse_weight = get_rmse(df_42_build['weight'], df_42_build['weight_fit'])
mape_weight = get_mape(df_42_build['weight'], df_42_build['weight_fit'])
# 日增重
r2_day_weight = get_r2(df_42_build['day_weight'], df_42_build['day_weight_fit'])
rmse_day_weight = get_rmse(df_42_build['day_weight'], df_42_build['day_weight_fit'])
mape_day_weight = get_mape(df_42_build['day_weight'], df_42_build['day_weight_fit'])

print(f"体重拟合评估：R2={r2_weight:.4f}, RMSE={rmse_weight:.4f}, MAPE={mape_weight:.4f}")
print(f"日增重拟合评估：R2={r2_day_weight:.4f}, RMSE={rmse_day_weight:.4f}, MAPE={mape_day_weight:.4f}")

体重拟合评估：R2=0.9999, RMSE=0.2878, MAPE=0.6650
日增重拟合评估：R2=0.8721, RMSE=0.0799, MAPE=9.4602


In [ ]:
# df_42_build.to_excel(PATH_DATA / 'growth' / '汇兴4栋生长数据-log-log拟合.xlsx', index=False)
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['weight'], mode='markers', name='实际重量', opacity=0.5))
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['weight_fit'], mode='lines', name='拟合重量', opacity=1))

change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green", annotation=dict(text='换料'))   

fig.update_layout(title='育肥4栋2单元日增重拟合') 
# fig.write_html(PATH_FIGURE_HTML/'育肥4栋2单元总重拟合.html')
fig.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['day_weight'], mode='markers', name='实际日增重'))
fig.add_trace(go.Scatter(x=df_42_build['age'], y=df_42_build['day_weight_fit'], mode='lines', name='拟合日增重'))

change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green", annotation=dict(text='换料'))   

fig.update_layout(title='育肥4栋2单元日增重拟合')

有点过拟合嫌疑

### 日龄-体重百分比拟合

In [ ]:
from feed_analysis.growing_fit.gompertz import fit_gompertz, gompertz_pred, fit_gompertz_fixed_L, fit_gompertz_fixed_x0

x = df_42_build['age'].values
y = df_42_build['day_pct_weight'].values
gompertz_params_42 = fit_gompertz(x, y)     # Gompertz模型拟合(直接最小二乘)
# gompertz_params_42 = fit_gompertz_fixed_L(x, y, 200)     # Gompertz模型拟合，固定L，R2最大可达0.7
# gompertz_params_42 = fit_gompertz_fixed_x0(x, y, x0=150)     # Gompertz模型拟合(直接最小二乘)
print(gompertz_params_42)

# 添加拟合结果
df_42_build['day_pct_weight_fit'] = gompertz_pred(df_42_build['age'].values, gompertz_params_42)  # 拟合结果
df_42_build['day_weight_fit'] = df_42_build['weight_fit'].diff()  # 日增重拟合结果
df_42_build.dropna(inplace=True)  # 日增重拟合结果

df_42_build.to_excel(PATH_DATA / 'growth' / '汇兴4栋生长数据-gompertz拟合.xlsx', index=False)

{'L': np.float64(0.06637191599884615), 'k': np.float64(2.0673333415706929e-16), 'x0': np.float64(240.06682416689472), 'cov': array([[3.18848063e-04, 2.88957004e-05, 0.00000000e+00],
       [2.88957004e-05, 2.71986621e-06, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00]])}


## 3栋数据

三栋总体数据&存栏量

In [ ]:
# 数据载入
import pandas as pd
from feed_analysis.config.path import PATH_DATA, PATH_FEED_PROCESSED
from feed_analysis.config.coding_schema import STD_HEADER_NAME
from feed_analysis.feed_pipeline.utils.age import recalibrate_age

# 读取喂食量数据 
df_31_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥3-1_build.parquet', engine='pyarrow')
df_32_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥3-2_build.parquet', engine='pyarrow')
df_33_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥3-3_build.parquet', engine='pyarrow')
df_34_build = pd.read_parquet(PATH_FEED_PROCESSED/'育肥3-4_build.parquet', engine='pyarrow')
# 读取存栏量数据
df_3_num = pd.read_excel(PATH_DATA / 'ori' / '育肥3栋总存栏数据.xlsx').rename(columns=STD_HEADER_NAME).sort_values(by=['Date'], ascending=True)
df_3_num['Date'] = df_3_num['Date']

# 计算头均值
df_3_build = pd.concat([df_31_build['food_total_kg'], df_32_build['food_total_kg'], df_33_build['food_total_kg'], df_34_build['food_total_kg']], axis=1)
df_3_build['total'] = df_3_build.sum(axis=1)
df_3_build.columns = ['3_1_feed', '3_2_feed', '3_3_feed', '3_4_feed', 'food_total_sum']

df_3_build['Date'] = df_31_build['Date']
df_3_build['age'] = df_31_build['age']

df_3_build['stock_num'] = df_3_build['Date'].map(df_3_num.set_index('Date')['stock_num'])
df_3_build['avg_food_kg'] = df_3_build['food_total_sum'] / df_3_build['stock_num']

# # 筛选日期
df_3_build = df_3_build.loc[df_3_build['Date'].between(pd.to_datetime('2025-08-28').date(), pd.to_datetime('2025-12-30').date()), :] 

# 插补缺失值
from feed_analysis.feed_pipeline.utils.interpolate import interpolation
df_3_build = interpolation(df_3_build, index_col='age', )

In [ ]:
from feed_analysis.growing_fit.log_log_fit import poly_log_regression_with_smearing
df_3_build, model_fit_3build, poly, scaler = poly_log_regression_with_smearing(df_3_build, x_col="age", y_col="avg_food_kg", degree=2, alpha=0.15, show_metrics=False)        # R2 0.972
df_3_build.to_excel(PATH_DATA / 'growth' / '汇兴3栋生长数据-log-log拟合.xlsx', index=False)

from feed_analysis.growing_fit.log_log_fit import plot_true_pred

fig = plot_true_pred(df_3_build,  x_name='age', y_true_name='avg_food_kg', y_pred_name='food_fit', plot_title='育肥3栋生长曲线拟合')
# 添加饲料的更换影响
change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green",annotation=dict(text='换料'))   
fig.write_html(PATH_FIGURE_HTML/'三栋喂食量时序图.html')
fig.show()

ImportError: cannot import name 'poly_log_regression_with_smearing' from 'feed_analysis.growing_fit.log_log_fit' (D:\Work\work-error-detect\src\feed_analysis\growing_fit\log_log_fit.py)

In [ ]:
import plotly.graph_objects as go
from feed_analysis.growing_fit.log_log_fit import poly_log_regression_with_smearing
df_3_build, model_fit_3build, poly, scaler = poly_log_regression_with_smearing(df_3_build, x_col="age", y_col="avg_food_kg", degree=2, alpha=0.15, show_metrics=False)        # R2 0.972
from feed_analysis.growing_fit.log_log_fit import plot_true_pred

# 添加四栋的数据
df_3_build['compare_4_true'] = df_3_build['age'].map(df_42_build.set_index('age')['avg_food_kg'])
df_3_build['compare_4_fit'] = df_3_build['age'].map(df_42_build.set_index('age')['food_fit'])

# fig = plot_true_pred(df_3_build,  x_name='age', y_true_name='avg_food_kg', y_pred_name='food_fit', plot_title='育肥3栋生长曲线拟合')
# 添加3栋数据
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_3_build['age'], y=df_3_build['avg_food_kg'], mode='markers', name='三栋真实值', opacity=1))
fig.add_trace(go.Scatter(x=df_3_build['age'], y=df_3_build['food_fit'], mode='lines', name='三栋拟合值', opacity=0.4))
# 添加饲料的更换影响
change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green",annotation=dict(text='换料'))  

# 添加4栋数据
fig.add_trace(go.Scatter(x=df_3_build['age'], y=df_3_build['compare_4_true'], mode='markers', name='四栋真实值', opacity=1))
fig.add_trace(go.Scatter(x=df_3_build['age'], y=df_3_build['compare_4_fit'], mode='lines', name='四栋拟合值', opacity=0.4))
fig.write_html(PATH_FIGURE_HTML / '三栋四栋对比喂食量时序图.html')
fig.show()

KeyError: 'food_fit'

### vs标准采食量

In [ ]:

import pandas as pd

# 读取标准采食量数据
df_std = pd.read_excel(PATH_DATA / 'ori' / '汇兴牧业-标准日龄饲料.xlsx', header=1)

df_3_build['std_feed'] = df_3_build['age'].map(df_std.set_index('日龄')['日喂料量(kg/头)'])

from feed_analysis.growing_fit.log_log_fit import plot_true_pred

fig = plot_true_pred(df_3_build,  x_name='age', y_true_name='avg_food_kg', y_pred_name='std_feed', plot_title='育肥3栋喂食vs2026汇兴牧业标准')
# 添加饲料的更换影响
change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green",annotation=dict(text='换料'))   
fig.write_html(PATH_FIGURE_HTML/'育肥3栋喂食vs标准.html')
fig.show()

c:\Users\yixua\.conda\envs\env\Lib\site-packages\plotly\io\_json.py:558: UserWarning:

Discarding nonzero nanoseconds in conversion.



In [ ]:

import pandas as pd

# 读取标准采食量数据
df_std = pd.read_excel(PATH_DATA / 'ori' / '汇兴牧业-标准日龄饲料.xlsx', header=1)

df_42_build['std_feed'] = df_42_build['age'].map(df_std.set_index('日龄')['日喂料量(kg/头)'])

from feed_analysis.growing_fit.log_log_fit import plot_true_pred
fig = plot_true_pred(df_42_build,  x_name='age', y_true_name='avg_food_kg', y_pred_name='std_feed', plot_title='育肥4栋2单元喂食vs2026汇兴牧业标准')
# 添加饲料的更换影响
change_feed_day = [25, 39, 53, 70, 97,] #  175, 180
for day in change_feed_day:
    fig.add_vline(x=day, line_width=2, opacity=0.4, line_color="green",annotation=dict(text='换料'))   
fig.write_html(PATH_FIGURE_HTML/'育肥4栋2单元喂食vs2026汇兴牧业标准.html')
fig.show()

### 结论

**基于标准对比**
- 目前在不限量的情况下，当前2026牧原标准低于猪只日饲料量
- 推荐提高喂食标准，以促进猪只生长

**三栋四栋对比**
- 70天的换料会有一个明显的进食料降低
- 四栋转舍日龄23，三栋转舍日龄32，而四栋在转舍7天之后出现了明显的下降趋势，结合[《母猪场决定小猪命运》](https://mp.weixin.qq.com/s/06qmSumNojJQ5b8CyW3Ltg)文章，可能是**断奶过早导致的免疫力降低**，在七天时处于一个“免疫力底下”时期。

**未来分析路径**
- 考虑使用拟合值 + 给定饲料的料肉比，给出日增重曲线
    